# Day 8 Project Solution: Support-Ticket Classifier

All five Day-8 concepts demonstrated in one working pipeline:
- **Lesson 1** — Zero-shot classification: label list + constrained-output prompt
- **Lesson 2** — `CLASSIFY_SYSTEM_PROMPT` constant with `{labels}` placeholder
- **Lesson 3** — `ClassificationResult` Pydantic schema (label + reasoning + confidence)
- **Lesson 4** — Label validation against the allowed set; fallback to `"Other"`
- **Lesson 5** — `classify_batch`: error-safe loop with Day-5 logging

## Imports and Constants

In [ ]:
import json
import logging
import ollama
from pydantic import BaseModel, Field, ValidationError

# Configure logging once at the entry point — never inside library functions
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(name)s  %(message)s",
)
logger = logging.getLogger(__name__)

MODEL = "llama3.2"

# Fixed label set — all tickets must land in exactly one of these
LABELS = ["Billing", "Technical", "Account", "Feature Request", "Other"]

# Ten realistic support tickets spanning all five categories
TICKETS = [
    "I was charged twice for my subscription this month and need a refund.",
    "The app keeps crashing every time I try to open the settings screen.",
    "I forgot my password and the reset email is not arriving in my inbox.",
    "It would be amazing if you could add a dark mode to the dashboard.",
    "My invoice shows a different amount than what I agreed to in the contract.",
    "The API is returning 500 errors intermittently on the /users endpoint.",
    "I need to update the email address associated with my account.",
    "Could you add the ability to export reports as CSV? That would save us hours.",
    "Just wanted to say thank you — your support team resolved my issue super fast!",
    "My two-factor authentication codes are not working and I am locked out.",
]

## Lesson 3 — `ClassificationResult` Pydantic Schema

Day-4 pattern: `BaseModel` + `Field(description=...)` gives us type-safe
parsing and a self-documenting schema we can embed in the system prompt.

In [ ]:
class ClassificationResult(BaseModel):
    """Structured output from the classifier.

    Attributes:
        label:      The chosen category from the allowed label list.
        reasoning:  One sentence explaining why this label was chosen.
        confidence: Self-reported confidence from 0.0 (uncertain) to 1.0 (certain).
    """

    label: str = Field(description="Exactly one label from the provided list")
    reasoning: str = Field(description="One sentence explaining why this label fits")
    confidence: float = Field(
        description="Confidence score between 0.0 (uncertain) and 1.0 (certain)"
    )


# Confirm the schema serialises correctly — this string goes into the system prompt
schema_str = json.dumps(ClassificationResult.model_json_schema(), indent=2)
print(schema_str)

## Lessons 1 + 2 — `CLASSIFY_SYSTEM_PROMPT` Constant

Module-level constant (not inline), with `{labels}` and `{schema}` placeholders.
A concrete example output prevents the schema-echo failure mode from Day 4.

In [ ]:
# Two-part structure from Lesson 2:
#   Part 1 — enumerate the exact allowed labels
#   Part 2 — constrain output to a single JSON object; forbid prose
CLASSIFY_SYSTEM_PROMPT = """You are a support-ticket classifier.

Classify the ticket into exactly one of these labels:
{labels}

Reply with a single JSON object that matches this schema exactly.
No prose, no markdown, no explanation outside the JSON.

Schema:
{schema}

Example output (for a billing question):
{{"label": "Billing", "reasoning": "The ticket mentions a charge and a refund.", "confidence": 0.95}}

Rules:
- label must be exactly one of the allowed labels listed above.
- reasoning must be a single sentence.
- confidence must be a number between 0.0 and 1.0.
- If unsure, pick the closest match and set confidence below 0.6."""

## Lessons 1 + 3 + 4 — `classify_ticket`

Single-item classifier. Demonstrates:
- Constrained-output prompting (Lesson 1 + 2)
- `format="json"` JSON mode from Day 4 (Lesson 3)
- Pydantic parsing with graceful `ValidationError` fallback (Lesson 3)
- Label validation against the allowed set (Lesson 4)

In [ ]:
def classify_ticket(
    text: str,
    labels: list[str] = LABELS,
    model: str = MODEL,
) -> ClassificationResult | None:
    """Classify a single support ticket.

    Args:
        text:   The ticket text to classify.
        labels: The allowed label set.
        model:  Ollama model name.

    Returns:
        A ClassificationResult with label, reasoning, and confidence,
        or None if JSON parsing fails (failure is logged).
    """
    # Build system prompt: inject label list and schema (Lessons 1, 2, 3)
    label_str = ", ".join(labels)
    schema_json = json.dumps(ClassificationResult.model_json_schema(), indent=2)
    system_msg = CLASSIFY_SYSTEM_PROMPT.format(labels=label_str, schema=schema_json)

    messages = [
        {"role": "system", "content": system_msg},
        {
            "role": "user",
            "content": (
                f"Support ticket:\n{text}\n\n"
                f"Classify into exactly one of: {label_str}"
            ),
        },
    ]

    logger.debug("classify_ticket() called: %r", text[:60])

    # format="json" enables grammar-constrained sampling (Day 4 pattern)
    response = ollama.chat(model=model, messages=messages, format="json")
    raw = response["message"]["content"]

    # Parse and validate the structured output (Lesson 3 graceful fallback)
    try:
        result = ClassificationResult.model_validate_json(raw)
    except ValidationError as e:
        logger.error("Pydantic validation failed: %s | raw=%r", e, raw[:200])
        return None

    # Lesson 4: validate the label is actually in the allowed set
    # Models occasionally capitalise differently or invent a new label
    allowed_lower = {lb.lower(): lb for lb in labels}
    canonical = allowed_lower.get(result.label.strip().lower())
    if canonical is None:
        logger.warning(
            "Model returned label %r not in allowed set %r — defaulting to 'Other'",
            result.label,
            labels,
        )
        result = ClassificationResult(
            label="Other",
            reasoning=result.reasoning,
            confidence=result.confidence,
        )
    else:
        # Normalise to the canonical capitalisation from LABELS
        result = ClassificationResult(
            label=canonical,
            reasoning=result.reasoning,
            confidence=result.confidence,
        )

    logger.info(
        "classify_ticket() -> label=%r confidence=%.2f",
        result.label,
        result.confidence,
    )
    return result

## Lesson 5 — `classify_batch`

Error-safe batch loop: one failure never stops the job. Day-5 logging at
three levels — INFO for milestones, DEBUG per item, WARNING on caught failures.

In [ ]:
def classify_batch(
    texts: list[str],
    labels: list[str] = LABELS,
    model: str = MODEL,
) -> list[dict]:
    """Classify a list of ticket texts without stopping on individual failures.

    Args:
        texts:  List of ticket strings to classify.
        labels: The allowed label set.
        model:  Ollama model name.

    Returns:
        A list of dicts, one per input text. Each dict always has:
        'text', 'label', 'confidence', 'reasoning', 'status', 'error'.
    """
    results: list[dict] = []
    total = len(texts)
    # Opening INFO log: records job parameters for the audit trail
    logger.info("classify_batch starting: %d tickets, model=%r", total, model)

    for i, text in enumerate(texts):
        # DEBUG per item: enables tracing without cluttering INFO logs
        logger.debug("Item %d/%d: %r", i + 1, total, text[:60])
        try:
            result = classify_ticket(text, labels=labels, model=model)
            if result is None:
                # classify_ticket returns None when JSON parsing fails
                raise ValueError("classify_ticket returned None (JSON parse failure)")
            results.append({
                "text": text,
                "label": result.label,
                "confidence": result.confidence,
                "reasoning": result.reasoning,
                "status": "ok",
                "error": None,
            })
        except Exception as e:
            # WARNING not ERROR: a single failure is notable but handled
            logger.warning(
                "Item %d/%d failed (%r): %s", i + 1, total, text[:40], e
            )
            results.append({
                "text": text,
                "label": None,
                "confidence": None,
                "reasoning": None,
                "status": "error",
                "error": str(e),
            })

    succeeded = sum(1 for r in results if r["status"] == "ok")
    failed = total - succeeded
    # Closing INFO log: summary statistics visible at a glance
    logger.info(
        "classify_batch done: %d/%d succeeded, %d failed",
        succeeded, total, failed,
    )
    return results

## Run the Classifier

Classify all 10 tickets and print per-ticket results.

In [ ]:
results = classify_batch(TICKETS, LABELS)

print("\n" + "=" * 70)
print("CLASSIFICATION RESULTS")
print("=" * 70)

for i, r in enumerate(results, start=1):
    if r["status"] == "ok":
        print(f"\nTicket {i:02d}")
        print(f"  Text      : {r['text'][:65]}")
        print(f"  Label     : {r['label']}")
        print(f"  Confidence: {r['confidence']:.2f}")
        print(f"  Reasoning : {r['reasoning']}")
    else:
        print(f"\nTicket {i:02d}  [ERROR]")
        print(f"  Text  : {r['text'][:65]}")
        print(f"  Error : {r['error']}")

## Summary Table

Count tickets per category and display a formatted table.

In [ ]:
# Tally successful results by label; use dict.get with default 0
counts: dict[str, int] = {label: 0 for label in LABELS}
for r in results:
    if r["status"] == "ok" and r["label"] in counts:
        counts[r["label"]] += 1

failed_count = sum(1 for r in results if r["status"] == "error")

print("\n" + "=" * 35)
print("SUMMARY TABLE")
print("=" * 35)
print(f"{'Category':<20} {'Count':>5}")
print("-" * 27)
for label, count in counts.items():
    print(f"{label:<20} {count:>5}")
print("-" * 27)
print(f"{'Total classified':<20} {sum(counts.values()):>5}")
if failed_count:
    print(f"{'Failed (error)':<20} {failed_count:>5}")
print("=" * 35)

## Deliverable Confirmation

In [ ]:
# Verify every required piece of the deliverable is present

# 1. Correct number of results
assert len(results) == len(TICKETS), (
    f"Expected {len(TICKETS)} results, got {len(results)}"
)

# 2. Every result dict has all required keys
required_keys = {"text", "label", "confidence", "reasoning", "status", "error"}
for idx, r in enumerate(results):
    assert required_keys.issubset(r.keys()), (
        f"Result {idx} missing keys: {required_keys - r.keys()}"
    )

# 3. All successful results have valid labels from LABELS
ok_results = [r for r in results if r["status"] == "ok"]
for r in ok_results:
    assert r["label"] in LABELS, f"Label {r['label']!r} not in LABELS"

# 4. All successful results have a confidence score and reasoning
for r in ok_results:
    assert isinstance(r["confidence"], float), "confidence must be a float"
    assert 0.0 <= r["confidence"] <= 1.0, (
        f"confidence {r['confidence']} out of range [0, 1]"
    )
    assert isinstance(r["reasoning"], str) and len(r["reasoning"]) > 0, (
        "reasoning must be a non-empty string"
    )

# 5. Summary table was produced (counts dict covers all labels)
assert set(counts.keys()) == set(LABELS), "Summary table must cover all labels"

print(f"Tickets submitted   : {len(TICKETS)}")
print(f"Tickets classified  : {len(ok_results)}")
print(f"Tickets failed      : {len(results) - len(ok_results)}")
print(f"Labels used         : {', '.join(LABELS)}")
print()
print("Deliverable confirmed:")
print("- 10 support tickets classified into Billing / Technical / Account /")
print("  Feature Request / Other using Ollama llama3.2 at localhost:11434.")
print("- Each result includes: original text, label, confidence, reasoning.")
print("- Final summary table shows ticket count per category.")
print("- No paid API used.")
print()
print("Day 8 project complete.")